# Model design checks

Two design questions are answered with data before the final model is trained.

1. **Which features should the HMM use?** Feature extraction computes 12 features per window, but
   several of them repeat each other (rank correlations of 0.95 and above). A model that treats the
   features as independent counts repeated evidence more than once. Four feature sets are compared.
2. **How much does the assumed transition matrix matter?** The training recordings contain no
   switches between activities, so the transition probabilities are an assumption (a prior). The
   stay probability is varied to see how much the results depend on it.

Neither check uses unseen data. The first uses blocked cross-validation inside the 92 training
windows, and the second uses simulated noise. Both are indications, not test results.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))
from gaussian_hmm import GaussianHMM

FEATURES_DIR = Path("../data/features")
ACTIVITIES = ["Still", "Standing", "Walking", "Jumping"]
N_STATES = len(ACTIVITIES)
VARIANCE_FLOOR = 1e-3

normalized = pd.read_csv(FEATURES_DIR / "window_features_normalized.csv")
ALL_FEATURES = [c for c in normalized.columns if c not in ("file", "activity")]
windows = {
    state: normalized[normalized["activity"] == name].sort_values("file")[ALL_FEATURES]
    for state, name in enumerate(ACTIVITIES)
}

# the same prior as in the training notebook: 18 pretend stay counts and 2 pretend switch counts per row
TRANS_PRIOR = np.full((N_STATES, N_STATES), 2 / 3)
np.fill_diagonal(TRANS_PRIOR, 18.0)
START_PRIOR = np.ones(N_STATES)

FEATURE_SETS = {
    "all 12 features": ALL_FEATURES,
    "8 features (near-copies removed)": [
        "acc_std", "acc_corr", "acc_jerk_ratio", "acc_dom_freq", "acc_low_freq_share",
        "gyro_corr", "gyro_jerk_ratio", "gyro_dom_freq",
    ],
    "strength only (2 std features)": ["acc_std", "gyro_std"],
    "rhythm only (10, without the std features)": [c for c in ALL_FEATURES if c not in ("acc_std", "gyro_std")],
}
{name: len(cols) for name, cols in FEATURE_SETS.items()}

{'all 12 features': 12,
 '8 features (near-copies removed)': 8,
 'strength only (2 std features)': 2,
 'rhythm only (10, without the std features)': 10}

**What the output shows.** Four feature sets are compared: all 12 features, 8 features without the near-copies, the two standard deviations alone, and the 10 features without the standard deviations.

## 1. Which features?

**Blocked cross-validation.** The 23 windows of each activity are cut into five blocks of consecutive
windows. In each round one block per activity is held out, the model is trained on the other four
blocks (leaving a gap of one window on each side of the held-out block, so that neighbouring windows
do not leak into the test), and the held-out blocks of the four activities are glued into one
sequence with three real switches. This is done in both orders (Still, Standing, Walking, Jumping
and the reverse), so every cell below counts 184 decoded windows; one wrong window moves a
percentage by about 0.5 points.

Noise is added to the held-out windows (normalized units) to see how each feature set copes with
data that is harder than the clean training windows. It is simulated and applies equally to every
feature. Each cell compares decoding with the Viterbi algorithm (uses the sequence) against
choosing the best activity for each window alone.

In [2]:
FOLDS = np.array_split(np.arange(23), 5)


def consecutive_runs(indices):
    """Split a sorted list of window numbers into runs of consecutive numbers."""
    runs, current = [], [indices[0]]
    for index in indices[1:]:
        if index == current[-1] + 1:
            current.append(index)
        else:
            runs.append(current)
            current = [index]
    return runs + [current]


def cross_validate(columns, noise, seed=0):
    """Percentage of held-out windows decoded correctly: (with Viterbi, one window at a time)."""
    rng = np.random.default_rng(seed)
    right_sequence = right_alone = total = 0
    for held in FOLDS:
        keep = [i for i in range(23) if i < held[0] - 1 or i > held[-1] + 1]
        sequences, means, variances = [], [], []
        for state in range(N_STATES):
            training = windows[state][columns].to_numpy()[keep]
            means.append(training.mean(axis=0))
            variances.append(training.var(axis=0))
            sequences += [windows[state][columns].to_numpy()[run] for run in consecutive_runs(keep)]
        model = GaussianHMM(N_STATES, len(columns), variance_floor=VARIANCE_FLOOR)
        model.set_parameters(np.full(N_STATES, 0.25), TRANS_PRIOR / TRANS_PRIOR.sum(axis=1, keepdims=True),
                             means, variances)
        model.fit(sequences, max_iterations=100, tolerance=1e-4, trans_prior=TRANS_PRIOR, start_prior=START_PRIOR)
        for order in ([0, 1, 2, 3], [3, 2, 1, 0]):
            x = np.vstack([windows[state][columns].to_numpy()[held] for state in order])
            truth = np.concatenate([[state] * len(held) for state in order])
            if noise:
                x = x + rng.normal(scale=noise, size=x.shape)
            right_sequence += int((model.viterbi(x)[0] == truth).sum())
            right_alone += int((model.log_emission(x).argmax(axis=1) == truth).sum())
            total += len(truth)
    return 100 * right_sequence / total, 100 * right_alone / total


NOISE_LEVELS = [0.0, 0.3, 0.6]
results = {name: [cross_validate(cols, noise) for noise in NOISE_LEVELS] for name, cols in FEATURE_SETS.items()}
labels = [f"noise {noise}" for noise in NOISE_LEVELS]
with_sequence = pd.DataFrame({n: [r[0] for r in rs] for n, rs in results.items()}, index=labels).T.round(1)
per_window = pd.DataFrame({n: [r[1] for r in rs] for n, rs in results.items()}, index=labels).T.round(1)
print("Windows decoded correctly (%), Viterbi:")
display(with_sequence)
print("Windows decoded correctly (%), best activity for each window alone:")
display(per_window)

Windows decoded correctly (%), Viterbi:


,noise 0.0,noise 0.3,noise 0.6
all 12 features,96.7,84.8,62.0
8 features (near-copies removed),96.7,89.7,72.3
strength only (2 std features),98.9,84.2,54.9
"rhythm only (10, without the std features)",91.8,78.3,59.2


Windows decoded correctly (%), best activity for each window alone:


,noise 0.0,noise 0.3,noise 0.6
all 12 features,96.7,83.2,63.6
8 features (near-copies removed),96.7,89.1,71.7
strength only (2 std features),98.9,82.6,57.6
"rhythm only (10, without the std features)",88.0,77.2,59.8


**Reading the table.** Removing the near-copies (8 features) is clearly better than keeping all 12
once noise is added, which fits the idea that repeated features count the same evidence several
times. On clean data the two standard-deviation features alone do best, but they collapse under
noise, so the rhythm features add robustness. The sequence model is not clearly better than
choosing each window alone on this data. The 8-feature set is used for the final model. It still
contains three time-domain feature types (standard deviation, axis correlation, jerk ratio) and two
frequency-domain feature types (dominant frequency, low-frequency share).

## 2. How much does the assumed stay probability matter?

The real training windows are glued into sequences that switch every four windows, noise is added,
and the sequences are decoded with models that differ only in the probability of staying in the
same activity. The emission parameters come from the labeled windows of the 8 chosen features.
This is a simulation with artificial noise: it shows how sensitive the decoding is to the
assumption, not how well the model performs on real unseen data.

In [3]:
CHOSEN = FEATURE_SETS["8 features (near-copies removed)"]
emission_means = np.array([windows[s][CHOSEN].mean().to_numpy() for s in range(N_STATES)])
emission_variances = np.array([windows[s][CHOSEN].var(ddof=0).to_numpy() for s in range(N_STATES)])
pool = {s: windows[s][CHOSEN].to_numpy() for s in range(N_STATES)}


def model_with_stay(stay):
    """The emission parameters of the chosen features with a given probability of staying."""
    trans = np.full((N_STATES, N_STATES), (1 - stay) / (N_STATES - 1))
    np.fill_diagonal(trans, stay)
    model = GaussianHMM(N_STATES, len(CHOSEN), variance_floor=VARIANCE_FLOOR)
    model.set_parameters(np.full(N_STATES, 0.25), trans, emission_means, emission_variances)
    return model


def simulate(noise, stays, repetitions=200, block=4, seed=0):
    """Percentage decoded correctly for each stay probability, and for each window alone."""
    rng = np.random.default_rng(seed)
    models = {stay: model_with_stay(stay) for stay in stays}
    right = {stay: 0 for stay in stays}
    alone = total = 0
    for repetition in range(repetitions):
        order = [0, 1, 2, 3] if repetition % 2 == 0 else [3, 2, 1, 0]
        blocks = [pool[s][rng.choice(23, size=block, replace=False)] for s in order]
        x = np.vstack(blocks)
        x = x + rng.normal(scale=noise, size=x.shape)
        truth = np.repeat(order, block)
        total += len(truth)
        alone += int((models[stays[0]].log_emission(x).argmax(axis=1) == truth).sum())
        for stay in stays:
            right[stay] += int((models[stay].viterbi(x)[0] == truth).sum())
    return {"each window alone": 100 * alone / total, **{f"stay {s:g}": 100 * right[s] / total for s in stays}}


STAYS = [0.5, 0.7, 0.8, 0.9, 0.9524, 0.99, 0.999]
sensitivity = pd.DataFrame({f"noise {n}": simulate(n, STAYS) for n in [0.0, 0.2, 0.4, 0.6, 0.8]}).T.round(1)
sensitivity

,each window alone,stay 0.5,stay 0.7,stay 0.8,stay 0.9,stay 0.9524,stay 0.99,stay 0.999
noise 0.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
noise 0.2,96.2,96.3,96.4,96.6,96.6,96.8,97.0,97.2
noise 0.4,85.8,85.9,86.2,86.2,86.3,86.5,86.5,86.0
noise 0.6,77.3,77.2,77.2,77.2,77.2,77.3,77.3,77.1
noise 0.8,69.6,69.6,69.6,69.8,69.7,69.8,69.9,69.4


**What the output shows.** At every noise level, the accuracy of the decoded windows changes by at most about 1 point across stay probabilities from 0.5 to 0.999 (for example 96.3 to 97.2 at noise 0.2 and 85.9 to 86.5 at noise 0.4). A stay probability of 0.5 gives almost the same result as choosing each window alone, and the best stay probability gains about 1 point over that. So in this simulation the assumed transition probabilities matter little, because the emission probabilities separate the activities well. This is a simulation with artificial noise, not a test on real data.

## 3. Which activities are confused?

The same blocked cross-validation as in section 1 for the 8 chosen features, with the confusion matrix of the Viterbi decoding added up over all rounds and both orders (184 windows, 46 per activity). The first check compares the total with the number in the table of section 1.

In [4]:
def cv_run(win, columns, noise, seed=0):
    """Blocked cross-validation as in section 1; returns the confusion matrix of the Viterbi decoding."""
    rng = np.random.default_rng(seed)
    matrix = np.zeros((N_STATES, N_STATES), dtype=int)
    for held in FOLDS:
        keep = [i for i in range(23) if i < held[0] - 1 or i > held[-1] + 1]
        sequences, means, variances = [], [], []
        for state in range(N_STATES):
            training = win[state][columns].to_numpy()[keep]
            means.append(training.mean(axis=0))
            variances.append(training.var(axis=0))
            sequences += [win[state][columns].to_numpy()[run] for run in consecutive_runs(keep)]
        model = GaussianHMM(N_STATES, len(columns), variance_floor=VARIANCE_FLOOR)
        model.set_parameters(np.full(N_STATES, 0.25), TRANS_PRIOR / TRANS_PRIOR.sum(axis=1, keepdims=True),
                             means, variances)
        model.fit(sequences, max_iterations=100, tolerance=1e-4, trans_prior=TRANS_PRIOR, start_prior=START_PRIOR)
        for order in ([0, 1, 2, 3], [3, 2, 1, 0]):
            x = np.vstack([win[state][columns].to_numpy()[held] for state in order])
            truth = np.concatenate([[state] * len(held) for state in order])
            if noise:
                x = x + rng.normal(scale=noise, size=x.shape)
            for t, p in zip(truth, model.viterbi(x)[0]):
                matrix[t, p] += 1
    return matrix


CHOSEN = FEATURE_SETS["8 features (near-copies removed)"]
confusions = {noise: cv_run(windows, CHOSEN, noise) for noise in NOISE_LEVELS}
for noise, matrix in confusions.items():
    table_value = with_sequence.loc["8 features (near-copies removed)", f"noise {noise}"]
    assert abs(100 * np.trace(matrix) / matrix.sum() - table_value) < 0.05, noise
print("Check passed: the totals equal the 8-feature row of the table in section 1.")
recall = pd.DataFrame({f"noise {n}": np.diag(m) / m.sum(axis=1) * 100 for n, m in confusions.items()}, index=ACTIVITIES).round(1)
display(recall)
for noise in (0.3, 0.6):
    print(f"Confusion matrix at noise {noise} (rows: true activity, columns: decoded activity)")
    display(pd.DataFrame(confusions[noise], index=ACTIVITIES, columns=ACTIVITIES))

Check passed: the totals equal the 8-feature row of the table in section 1.


,noise 0.0,noise 0.3,noise 0.6
Still,91.3,95.7,91.3
Standing,100.0,100.0,84.8
Walking,100.0,73.9,34.8
Jumping,95.7,89.1,78.3


Confusion matrix at noise 0.3 (rows: true activity, columns: decoded activity)


,Still,Standing,Walking,Jumping
Still,44,2,0,0
Standing,0,46,0,0
Walking,0,11,34,1
Jumping,0,3,2,41


Confusion matrix at noise 0.6 (rows: true activity, columns: decoded activity)


,Still,Standing,Walking,Jumping
Still,42,4,0,0
Standing,1,39,6,0
Walking,0,25,16,5
Jumping,0,8,2,36


**What the output shows.** The check passes, so the new function reproduces the table of section 1. Without noise, Standing and Walking are always right, Still is right in 91.3% and Jumping in 95.7% of its windows. With noise 0.3, Walking drops to 73.9% (11 of its 46 windows are decoded as Standing and 1 as Jumping) while Standing stays at 100%; with noise 0.6, Walking is right in only 34.8% of its windows (25 are decoded as Standing). Standing is also the activity most often given to Still and Jumping windows when the model is wrong (2 and 3 windows at noise 0.3, 4 and 8 at noise 0.6). So Walking is the hardest activity to keep correct as the data gets noisier, and Standing is where the model's mistakes end up; the reason for this was not tested. The noise is simulated: Gaussian noise of the same size added to every normalized feature.

## 4. Does the sampling rate matter?

The 50 Hz readings of every window are thinned to 25 Hz (every 2nd reading) and 10 Hz (every 5th reading), the 12 features are computed again from the thinned window, logarithm and Z-score are fitted on the 92 thinned windows, and the cross-validation of section 1 is repeated for the 8 chosen features. The first row (every reading, 50 Hz) is a check against the table in section 1.

In [5]:
from window_features import log_transform, window_features

PROCESSED_DIR = Path("../data/processed")


def thinned_windows(step):
    """The 8 chosen features of every training window, computed from every `step`-th reading."""
    rows = []
    for activity in ACTIVITIES:
        for path in sorted((PROCESSED_DIR / activity.lower()).glob("*.csv")):
            window = pd.read_csv(path).iloc[::step].reset_index(drop=True)
            rows.append({"activity": activity, "file": path.name, **window_features(window)})
    table = pd.DataFrame(rows)
    scaled = log_transform(table[ALL_FEATURES])
    scaled = (scaled - scaled.mean()) / scaled.std(ddof=0)
    scaled[["activity", "file"]] = table[["activity", "file"]]
    return {s: scaled[scaled["activity"] == a].sort_values("file")[ALL_FEATURES] for s, a in enumerate(ACTIVITIES)}


rows = []
for step, rate in [(1, 50), (2, 25), (5, 10)]:
    win = thinned_windows(step)
    row = {"sampling rate (Hz)": rate, "readings per window": len(pd.read_csv(PROCESSED_DIR / "still" / "still_01.csv").iloc[::step])}
    for noise in (0.0, 0.3):
        matrix = cv_run(win, CHOSEN, noise)
        row[f"windows right (%), noise {noise}"] = round(100 * np.trace(matrix) / matrix.sum(), 1)
    rows.append(row)
table_check = with_sequence.loc["8 features (near-copies removed)"]
assert abs(rows[0]["windows right (%), noise 0.0"] - table_check["noise 0.0"]) < 0.05
assert abs(rows[0]["windows right (%), noise 0.3"] - table_check["noise 0.3"]) < 0.05
print("Check passed: the 50 Hz row equals the 8-feature row of the table in section 1.")
pd.DataFrame(rows).set_index("sampling rate (Hz)")

Check passed: the 50 Hz row equals the 8-feature row of the table in section 1.


,readings per window,"windows right (%), noise 0.0","windows right (%), noise 0.3"
sampling rate (Hz),,,
50,255,96.7,89.7
25,128,96.7,88.0
10,51,98.9,84.8


**What the output shows.** Thinning to 25 Hz (128 readings per window) leaves the share of correctly decoded windows at 96.7% without noise and lowers it from 89.7% to 88.0% with noise 0.3. Thinning to 10 Hz (51 readings) gives 98.9% without noise and 84.8% with noise 0.3. One window is 0.54 points, so the differences without noise are 0 to 4 windows out of 184, and with noise 0.3 they are 3 and 9 windows. Within this range the sampling rate matters little. Two limits: the thinning keeps every n-th reading without filtering, so at 10 Hz fast movement can be folded into slower frequencies (the highest dominant frequency of a Jumping window is 4.89 Hz, close to the 5 Hz limit of 10 Hz sampling), and this is a simulation on one phone, not a comparison of different phones.